# Lecture 5: Performance Measurement and Regularization


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score

np.random.seed(42)
print("Libraries loaded.")


## 1. Bias-Variance Tradeoff (Ch. 8.2)

$$\\text{Error} = \\underbrace{\\text{Bias}^2}_{\\text{underfitting}} + \\underbrace{\\text{Variance}}_{\\text{overfitting}} + \\underbrace{\\text{Noise}}_{\\text{irreducible}}$$

- **High bias (underfitting)**: Model too simple → high train and test loss
- **High variance (overfitting)**: Model too complex → low train, high test loss


In [ ]:
# Bias-variance tradeoff as polynomial degree increases
np.random.seed(0)
N_train = 30
x_all = np.linspace(-3, 3, 200)
y_true_fn = lambda x: np.sin(x) + 0.3*x

x_train = np.random.uniform(-3, 3, N_train)
y_train = y_true_fn(x_train) + np.random.randn(N_train) * 0.4
x_test  = np.random.uniform(-3, 3, 100)
y_test  = y_true_fn(x_test)  + np.random.randn(100) * 0.4

degrees = [1, 3, 6, 12]
train_errs, test_errs = [], []

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, deg in zip(axes, degrees):
    pipe = Pipeline([('poly', PolynomialFeatures(deg)), ('lr', LinearRegression())])
    pipe.fit(x_train.reshape(-1,1), y_train)
    
    y_pred_train = pipe.predict(x_train.reshape(-1,1))
    y_pred_test  = pipe.predict(x_test.reshape(-1,1))
    
    tr_err = np.mean((y_pred_train - y_train)**2)
    te_err = np.mean((y_pred_test  - y_test)**2)
    train_errs.append(tr_err); test_errs.append(te_err)
    
    y_plot = pipe.predict(x_all.reshape(-1,1))
    ax.scatter(x_train, y_train, s=20, alpha=0.6, color='gray', label="Train")
    ax.plot(x_all, y_true_fn(x_all), 'g--', lw=1.5, label="True")
    ax.plot(x_all, y_plot, 'crimson', lw=2, label="Model")
    ax.set_ylim(-3, 3)
    ax.set_title(f"Degree {deg}\nTrain:{tr_err:.3f} Test:{te_err:.3f}")
    ax.legend(fontsize=7); ax.grid(alpha=0.3)

plt.suptitle("Polynomial Degree: Underfitting -> Overfitting", fontsize=13)
plt.tight_layout()
plt.savefig("fig_nb05_bias_var.png", dpi=100, bbox_inches='tight')
plt.show()

fig2, ax2 = plt.subplots(figsize=(8, 4))
ax2.plot(degrees, train_errs, '-o', color='steelblue', lw=2, label="Training Error")
ax2.plot(degrees, test_errs,  '-o', color='crimson',   lw=2, label="Test Error")
ax2.set_xlabel("Polynomial Degree"); ax2.set_ylabel("MSE")
ax2.set_title("Train vs Test Error (by Model Complexity)")
ax2.legend(); ax2.grid(alpha=0.3)
ax2.annotate("Underfitting", xy=(1, test_errs[0]), xytext=(1.3, test_errs[0]+0.05), fontsize=10, color='orange')
ax2.annotate("Overfitting",  xy=(12, test_errs[-1]), xytext=(9, test_errs[-1]+0.05), fontsize=10, color='red')
plt.tight_layout()
plt.savefig("fig_nb05_bias_var2.png", dpi=100, bbox_inches='tight')
plt.show()


## 2. Double Descent (Ch. 8.4)

In classical statistics, test error increases in a U-shape as model complexity grows.  
However, in large deep learning models a **second descent** is observed:  
once the model passes the interpolation threshold, test error decreases again.


In [ ]:
# Conceptual simulation of double descent
n_params = np.array([2, 5, 10, 20, 30, 40, 50, 75, 100, 150, 200, 300, 500])

def double_descent_curve(n, n_data=30, noise=0.3):
    if n < n_data:
        return 1.5 - 0.04*n + noise*np.random.randn()
    elif n < n_data * 1.2:
        t = (n - n_data) / (n_data * 0.2)
        return 0.7 + 2.5*np.exp(-t) + noise*np.random.randn()
    else:
        return 0.2 + 8.0/n + noise*np.random.randn()*0.3

np.random.seed(5)
test_errs_dd = [max(0.1, double_descent_curve(n)) for n in n_params]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(n_params, test_errs_dd, '-o', color='steelblue', lw=2, markersize=6)
ax.axvline(30, color='red', linestyle='--', lw=1.5, label="Interpolation threshold (~data size)")
ax.fill_betweenx([0, 3.5], 0, 30, alpha=0.08, color='orange', label="Classical regime")
ax.fill_betweenx([0, 3.5], 30, 500, alpha=0.08, color='blue', label="Modern (overparameterized)")
ax.set_xlabel("Number of Parameters (model complexity)", fontsize=12)
ax.set_ylabel("Test Error", fontsize=12)
ax.set_title("Double Descent Curve", fontsize=14)
ax.legend(fontsize=10); ax.grid(alpha=0.3)
ax.set_ylim(0, 3.5)
ax.annotate("Underfitting", xy=(5, 1.3), fontsize=10, color='orange')
ax.annotate("Interpolation\nThreshold", xy=(28, 2.8), fontsize=9, color='red')
ax.annotate("Overfitted model\ngeneralizes well", xy=(150, 0.4), fontsize=10, color='blue')
plt.tight_layout()
plt.savefig("fig_nb05_double_descent.png", dpi=100, bbox_inches='tight')
plt.show()


## 3. L2 Regularization (Weight Decay) — Ch. 9.1

$$L_{reg} = L_{data} + \\lambda \\|\\theta\\|^2$$

Large λ → parameters stay small → simpler model → less overfitting


In [ ]:
# Ridge (L2) vs Lasso (L1) vs Unregularized
np.random.seed(3)
from sklearn.datasets import make_regression

X_r, y_r = make_regression(n_samples=80, n_features=20, noise=15, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_r, y_r, test_size=0.3)

alphas = np.logspace(-3, 4, 50)
ridge_train, ridge_test = [], []
lasso_train, lasso_test = [], []

for a in alphas:
    r = Ridge(alpha=a).fit(X_tr, y_tr)
    ridge_train.append(np.mean((r.predict(X_tr)-y_tr)**2))
    ridge_test.append(np.mean((r.predict(X_te)-y_te)**2))
    
    l = Lasso(alpha=a, max_iter=5000).fit(X_tr, y_tr)
    lasso_train.append(np.mean((l.predict(X_tr)-y_tr)**2))
    lasso_test.append(np.mean((l.predict(X_te)-y_te)**2))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].semilogx(alphas, ridge_train, 'steelblue', lw=2, label="Train")
axes[0].semilogx(alphas, ridge_test,  'crimson',   lw=2, label="Test")
axes[0].set_title("Ridge (L2) — lambda vs Error")
axes[0].set_xlabel("lambda"); axes[0].set_ylabel("MSE")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].semilogx(alphas, lasso_train, 'steelblue', lw=2, label="Train")
axes[1].semilogx(alphas, lasso_test,  'crimson',   lw=2, label="Test")
axes[1].set_title("Lasso (L1) — lambda vs Error")
axes[1].set_xlabel("lambda"); axes[1].set_ylabel("MSE")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("fig_nb05_regularization.png", dpi=100, bbox_inches='tight')
plt.show()


## 4. Dropout (Ch. 9.3)

During training, neurons are randomly deactivated with probability p.  
This simulates an **ensemble** of different network architectures.  
At test time all neurons are active; weights are scaled by (1-p).


In [ ]:
def dropout_forward(h, p_drop=0.3, training=True):
    if not training:
        return h  # test: all neurons active
    mask = (np.random.rand(*h.shape) > p_drop).astype(float)
    return h * mask / (1 - p_drop)  # inverted dropout

# Dropout mask visualization
np.random.seed(7)
h_example = np.ones((5, 10))
p_values = [0.0, 0.3, 0.5, 0.7]

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, p in zip(axes, p_values):
    masked = dropout_forward(h_example, p_drop=p, training=True)
    active = (masked > 0).astype(int)
    ax.imshow(active, cmap='Blues', vmin=0, vmax=1, aspect='auto')
    active_pct = active.mean() * 100
    ax.set_title(f"Dropout p={p}\n{active_pct:.0f}% active neurons")
    ax.set_xlabel("Neurons"); ax.set_ylabel("Samples")
    for i in range(5):
        for j in range(10):
            ax.text(j, i, str(active[i,j]), ha='center', va='center', fontsize=8,
                   color='white' if active[i,j] else 'gray')

plt.suptitle("Dropout: Deactivating Neurons at Different Rates", fontsize=12)
plt.tight_layout()
plt.savefig("fig_nb05_dropout.png", dpi=100, bbox_inches='tight')
plt.show()


## 5. Batch Normalization (Ch. 9.3)

Normalizes activations within each mini-batch:

$$\\hat{x}_i = \\frac{x_i - \\mu_B}{\\sqrt{\\sigma_B^2 + \\epsilon}}, \\quad y_i = \\gamma \\hat{x}_i + \\beta$$

Keeping the input distribution stable at each layer → **faster training**, **larger learning rates** possible.


In [ ]:
def batch_norm_forward(x, gamma=1.0, beta=0.0, eps=1e-8):
    mu = x.mean(axis=0)
    var = x.var(axis=0)
    x_hat = (x - mu) / np.sqrt(var + eps)
    return gamma * x_hat + beta, mu, var

np.random.seed(2)
x_raw = np.random.randn(100, 1) * 5 + 10  # Poor: mean!=0, std!=1
x_bn, mu, var = batch_norm_forward(x_raw)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(x_raw, bins=25, color='crimson', edgecolor='white', alpha=0.8)
axes[0].set_title(f"Before BatchNorm\nMean={x_raw.mean():.1f}, Std={x_raw.std():.1f}")
axes[0].set_xlabel("Activation value")

axes[1].hist(x_bn, bins=25, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].set_title(f"After BatchNorm\nMean={x_bn.mean():.3f}, Std={x_bn.std():.3f}")
axes[1].set_xlabel("Normalized activation")

plt.suptitle("Batch Normalization: Activation Distribution Normalization", fontsize=12)
plt.tight_layout()
plt.savefig("fig_nb05_batchnorm.png", dpi=100, bbox_inches='tight')
plt.show()


## 6. Early Stopping

Stop training when the validation loss starts to increase.  
This is a de-facto regularization technique: it limits the number of training steps.


In [ ]:
# Early stopping simulation
np.random.seed(9)
epochs = np.arange(1, 201)
train_loss = 1.0 * np.exp(-0.02*epochs) + 0.03*np.random.randn(200)*0.5
val_loss   = 1.0 * np.exp(-0.015*epochs) + 0.1*(epochs/200)**1.5 + 0.05*np.random.randn(200)

from scipy.ndimage import uniform_filter1d
train_smooth = uniform_filter1d(np.abs(train_loss), size=5)
val_smooth   = uniform_filter1d(np.abs(val_loss),   size=5)

best_epoch = np.argmin(val_smooth)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs, train_smooth, 'steelblue', lw=2, label="Training loss")
ax.plot(epochs, val_smooth,   'crimson',   lw=2, label="Validation loss")
ax.axvline(best_epoch+1, color='green', linestyle='--', lw=2,
           label=f"Early stopping (epoch {best_epoch+1})")
ax.fill_betweenx([0, 1.2], best_epoch+1, 200, alpha=0.1, color='red', label="Unnecessary training")
ax.set_title("Early Stopping", fontsize=14)
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.legend(); ax.grid(alpha=0.3)
ax.set_ylim(0, 1.2)
plt.tight_layout()
plt.savefig("fig_nb05_early_stop.png", dpi=100, bbox_inches='tight')
plt.show()
print(f"Best epoch: {best_epoch+1}, val loss: {val_smooth[best_epoch]:.4f}")


## 7. Summary

| Technique | Type | Effect |
|---|---|---|
| **L2 (Ridge)** | Explicit reg. | Penalizes large weights |
| **L1 (Lasso)** | Explicit reg. | Sparse solution, feature selection |
| **Dropout** | Implicit reg. | Ensemble effect, prevents co-adaptation |
| **Batch Norm** | Heuristic | Speeds up training, allows larger lr |
| **Early Stopping** | Implicit reg. | Stop before overfitting |
| **Data Augmentation** | Implicit reg. | Increases data diversity |

**Next Notebook →** Convolutional Neural Networks (CNN) and ResNet
